In [2]:
import ifcopenshell
import ifcopenshell.util.element
import ifcopenshell.geom
import ifcopenshell.guid
from shapely.geometry import Polygon
from shapely.ops import unary_union
import json
from rdflib import Graph, Literal, Namespace, RDF, RDFS, OWL
import uuid
from pathlib import Path
from pprint import pprint

In [2]:
# Configuration of Namespaces for RDF graph
BRICK = Namespace("https://brickschema.org/schema/Brick#")
BOT = Namespace("https://w3id.org/bot#")
INST = Namespace("https://lbd.example.com/")
PROPS = Namespace("http://lbd.arch.rwth-aachen.de/props#") 
S223 = Namespace("http://data.ashrae.org/standard223#")

PREFIXES = {
    "brick": BRICK, "bot": BOT, "inst": INST, "rdfs": RDFS, "owl": OWL, "props": PROPS, "s223": S223
    }


In [ ]:
def initialize_graph():        
    g = Graph()
    for p, ns in PREFIXES.items(): g.bind(p, ns)
    return g

In [ ]:
def add_entity_instances(graph, ifc_model, entity_type, rdf_class):
    """
    Add entity instances from an IFC model to a graph.
    """            
    entities = ifc_model.by_type(entity_type)
    for entity in entities:
        entity_uuid = uuid.UUID(ifcopenshell.guid.expand(entity.GlobalId))
        entity_uri = INST[f"{entity_type.lower()}_{entity_uuid}"]
        graph.add((entity_uri, RDF.type, rdf_class))
        graph.add((entity_uri, RDFS.label, Literal(entity.Name)))
        graph.add((entity_uri, PROPS.descriptionIfcRoot_attribute_simple, Literal(entity.Description)))
        graph.add((entity_uri, PROPS.globalIdIfcRoot_attribute_simple, Literal(entity.GlobalId)))
    added_entities = len(entities)
    print(f"Added {added_entities} {entity_type.lower()}s from IFC model to the graph.")      

    return graph, added_entities

In [ ]:
relation_mapping = {
    "IfcRelAggregates": [("IfcSite", "IfcBuilding", BOT.hasBuilding),
                         ("IfcBuilding", "IfcBuildingStorey", BOT.hasStorey),
                          ("IfcBuildingStorey", "IfcSpace", BOT.hasSpace)],
    "IfcRelContainedInSpatialStructure": [("IfcSpace", "IfcElement", BOT.containsElement),
                                          ("IfcBuildingStorey", "IfcElement", BOT.containsElement)]
}

In [ ]:
ifctype_mapping = {
    "IfcSite": BOT.Site,
    "IfcBuilding": BOT.Building,
    "IfcBuildingStorey": BOT.BuildingStorey,
    "IfcSpace": BOT.Space,

In [ ]:
def ifc_relations_to_rdf(graph, ifc_model, relation_type, subject_class, object_class, predicate):
    """
    Convert IFC relationships to RDF triples in the graph.
    """    
    relations = ifc_model.by_type(relation_type)
    for rel in relations:
        if rel.is_a(relation_type):
            subject_entity = getattr(rel, "Relating" + subject_class)
            object_entities = getattr(rel, "Related" + object_class + "s")
            if subject_entity and object_entities:
                subject_uuid = uuid.UUID(ifcopenshell.guid.expand(subject_entity.GlobalId))
                subject_uri = INST[f"{subject_class.lower()}_{subject_uuid}"]
                for obj in object_entities:
                    obj_uuid = uuid.UUID(ifcopenshell.guid.expand(obj.GlobalId))
                    obj_uri = INST[f"{object_class.lower()}_{obj_uuid}"]
                    graph.add((subject_uri, predicate, obj_uri))
    print(f"Added {len(relations)} {relation_type} relationships to the graph.")
    return graph

In [ ]:
def add_relationships(graph, ifc_model, relationship_type, subject_class, object_class, predicate):
    """
    Add relationships from an IFC model to a graph.
    """    
    relationships = ifc_model.by_type(relationship_type)
    for relationship in relationships:
        if hasattr(relationship, relationship_type["subject"]) and hasattr(relationship, relationship_type["object"]):
            subject_entity = getattr(relationship, relationship_type["subject"])
            object_entities = getattr(relationship, relationship_type["object"])
            subject_uuid = uuid.UUID(ifcopenshell.guid.expand(subject_entity.GlobalId))
            subject_uri = INST[f"{subject_entity.is_a().lower()}_{subject_uuid}"]
            graph.add((subject_uri, RDF.type, subject_class))
            for object_entity in object_entities:
                object_uuid = uuid.UUID(ifcopenshell.guid.expand(object_entity.GlobalId))
                object_uri = INST[f"{object_entity.is_a().lower()}_{object_uuid}"]
                graph.add((object_uri, RDF.type, object_class))
                graph.add((subject_uri, predicate, object_uri))
    added_relationships = len(relationships)
    print(f"Added {added_relationships} {relationship_type.lower()} relationships from IFC model to the graph.")      

    return graph, added_relationships

In [3]:
arc_ifc = r"C:\Users\yanpe\OneDrive - Metropolia Ammattikorkeakoulu Oy\Research\MD2MV\data\IFC\01ARK\ARK_MET.ifc"
arc_ttl = r"C:\Users\yanpe\OneDrive - Metropolia Ammattikorkeakoulu Oy\Research\MD2MV\data\TTL\01ARK\ARK_MET.ttl"

In [4]:
# 1. Initialize an empty graph
g = Graph()

# 2. Parse the Turtle file
# Replace 'your_data.ttl' with the actual path to your file
g.parse(arc_ttl, format="turtle")

# 3. Create a set to store the unique types
unique_types = set()

# 4. Find all instances of rdf:type
# g.triples() takes a tuple of (Subject, Predicate, Object). 
# 'None' acts as a wildcard.
for subject, predicate, obj in g.triples((None, RDF.type, None)):
    unique_types.add(obj)

# 5. Print the results
print(f"Found {len(unique_types)} unique rdf:types:")
for rdf_type in unique_types:
    print(f" - {rdf_type}")

Found 7 unique rdf:types:
 - https://w3id.org/bot#Site
 - http://www.w3.org/2002/07/owl#ObjectProperty
 - http://www.w3.org/2002/07/owl#DatatypeProperty
 - https://w3id.org/bot#Element
 - https://w3id.org/bot#Building
 - https://w3id.org/bot#Storey
 - https://w3id.org/bot#Space


In [6]:
arc_ifc_model = ifcopenshell.open(arc_ifc)

In [10]:
building = arc_ifc_model.by_type("IfcBuilding")[0]
psets = ifcopenshell.util.element.get_psets(building)
pprint(psets)

{'Identity Data': {'Author': '',
                   'Building Name': '',
                   'CQ_DatabaseGUID': 'apps_qskxdm8v-iq1e-xmc4-nwog16fnenuuvxkz',
                   'CQ_ModelName': 'Metropolia',
                   'Edited by': '',
                   'Organization Description': '',
                   'Organization Name': '',
                   'Tontti / Rno': '3',
                   'Workset': 'Project Info',
                   'id': 23697096,
                   'kaupunginosa/kylä': '45',
                   'kortteli/tila': 45165.0},
 'Other': {'Category': 'Project Information',
           'Client Name': 'Owner',
           'Extensions.Parameters': '2016.5',
           'Project Address': 'Myllypurontie 1\r\n00920 Helsinki',
           'Project Issue Date': 'Issue Date',
           'Project Name': 'Metropolia Myllypuron kampus',
           'Project Number': 'MET',
           'Project Status': 'Project Status',
           'id': 23697101},
 'Pset_BuildingCommon': {'NumberOfStoreys